# 03 — Cell-type deconvolution: cell2location and Tangram

Visium spots are not single cells: each ~55 μm spot averages 1–10 cells depending on tissue density. To get cell-type maps we deconvolve against a sarcoma-tuned scRNA reference using **two methods** so you can compare:

- **cell2location** — a probabilistic Bayesian model that learns a count-based regression on the scRNA reference and then estimates per-spot cell counts. Conservative on rare types; gives uncertainty intervals.
- **Tangram** — a deep-learning approach that maps individual scRNA cells onto spatial spots by aligning marker-gene expression. Faster to fit; slightly more confident on rare types but can over-commit.

We show the same five cell types (malignant, CAF, endothelial, macrophage, T cell) under both methods and discuss where they agree and where they don't.

In [ ]:
sample      = "GSE227469_angiosarcoma_01"
data_dir    = "data"
results_dir = "results"

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt

spatial_dir = Path(results_dir) / "visium" / sample
ref_dir     = Path(data_dir) / "references"

adata = sc.read_h5ad(spatial_dir / "adata_normalized.h5ad")
adata

## cell2location

We train the regression model on the reference once (this is the slow step) and reuse the resulting signature matrix. For a real run, fit the model on a GPU; on CPU it takes ~30 minutes for the regression and ~1 hour for the spatial step.

In [ ]:
# Either load the pipeline-produced output ...
c2l_path = spatial_dir / "adata_c2l.h5ad"
if c2l_path.exists():
    adata_c2l = sc.read_h5ad(c2l_path)
    print("Loaded cell2location output.")
else:
    # ... or fit it here, inline.
    from cell2location.models import RegressionModel, Cell2location
    ref = sc.read_h5ad(ref_dir / "sarcoma_reference_signatures.h5ad")
    RegressionModel.setup_anndata(ref, labels_key="cell_type_l1",
                                  batch_key="dataset" if "dataset" in ref.obs else None)
    rmod = RegressionModel(ref)
    rmod.train(max_epochs=250, batch_size=2500, train_size=1, lr=0.002)
    ref  = rmod.export_posterior(ref, sample_kwargs={"num_samples": 1000,
                                                     "batch_size": 2500})
    sig  = ref.varm["means_per_cluster_mu_fg"].copy()
    sig.columns = [c.replace("means_per_cluster_mu_fg_", "") for c in sig.columns]

    shared = sig.index.intersection(adata.var_names)
    a = adata[:, shared].copy()
    Cell2location.setup_anndata(a)
    smod = Cell2location(a, cell_state_df=sig.loc[shared],
                         N_cells_per_location=10, detection_alpha=20)
    smod.train(max_epochs=15000, batch_size=None, train_size=1)
    adata_c2l = smod.export_posterior(a, sample_kwargs={"num_samples": 1000,
                                                        "batch_size": a.n_obs})
    adata_c2l.write(c2l_path)

In [ ]:
prop_c2l = adata_c2l.obsm["q05_cell_abundance_w_sf"].copy()
prop_c2l.columns = [c.replace("q05cell_abundance_w_sf_", "") for c in prop_c2l.columns]
prop_c2l = prop_c2l.div(prop_c2l.sum(axis=1), axis=0).fillna(0.0)
for col in prop_c2l.columns:
    adata_c2l.obs[f"fraction_{col}"] = prop_c2l[col].values
prop_c2l.head()

In [ ]:
panel = ["malignant", "caf", "endothelial", "macrophage", "t_cell"]
cols  = [f"fraction_{p}" for p in panel if f"fraction_{p}" in adata_c2l.obs.columns]
sc.pl.spatial(adata_c2l, color=cols, ncols=5, cmap="viridis", size=1.3)

## Tangram

Tangram maps individual scRNA cells onto spots, then aggregates labels per spot.

In [ ]:
tg_path = spatial_dir / "adata_tangram.h5ad"
if tg_path.exists():
    adata_tg = sc.read_h5ad(tg_path)
else:
    import tangram as tg
    sc_ref = sc.read_h5ad(ref_dir / "sarcoma_reference_combined.h5ad")
    sc.tl.rank_genes_groups(sc_ref, "cell_type_l1", method="wilcoxon", n_genes=100)
    markers = (sc.get.rank_genes_groups_df(sc_ref, group=None)
                 .groupby("group").head(100)["names"].unique().tolist())
    shared = [g for g in markers if g in adata.var_names]
    tg.pp_adatas(sc_ref, adata, genes=shared)
    ad_map = tg.map_cells_to_space(adata_sc=sc_ref, adata_sp=adata,
                                   device="cpu", mode="cells",
                                   density_prior="rna_count_based", num_epochs=500)
    tg.project_cell_annotations(ad_map, adata, annotation="cell_type_l1")
    pred = adata.obsm["tangram_ct_pred"].copy()
    pred = pred.div(pred.sum(axis=1), axis=0).fillna(0.0)
    for col in pred.columns:
        adata.obs[f"fraction_{col}"] = pred[col].values
    adata.write(tg_path)
    adata_tg = adata

In [ ]:
cols_tg = [f"fraction_{p}" for p in panel if f"fraction_{p}" in adata_tg.obs.columns]
sc.pl.spatial(adata_tg, color=cols_tg, ncols=5, cmap="viridis", size=1.3)

## Method comparison
Spot-level Spearman correlation per cell type. High correlation means the methods agree on the spatial distribution even if the absolute fractions differ.

In [ ]:
from scipy.stats import spearmanr
shared_spots = adata_c2l.obs_names.intersection(adata_tg.obs_names)
rows = []
for p in panel:
    col = f"fraction_{p}"
    if col in adata_c2l.obs and col in adata_tg.obs:
        a = adata_c2l.obs.loc[shared_spots, col]
        b = adata_tg.obs.loc[shared_spots,  col]
        r, _ = spearmanr(a, b)
        rows.append({"cell_type": p, "spearman_r": r,
                     "c2l_mean": a.mean(), "tg_mean": b.mean()})
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, p in enumerate(panel):
    col = f"fraction_{p}"
    sc.pl.spatial(adata_c2l, color=col, ax=axes[0, i], show=False,
                  title=f"c2l — {p}", cmap="viridis", size=1.3)
    sc.pl.spatial(adata_tg,  color=col, ax=axes[1, i], show=False,
                  title=f"tangram — {p}", cmap="viridis", size=1.3)
fig.tight_layout()
fig.savefig(Path(results_dir) / "figures" / f"{sample}_c2l_vs_tangram.pdf",
            bbox_inches="tight")

## Takeaways

- For the dominant compartments (malignant, CAF, endothelial), both methods produce visually similar maps and high Spearman correlation.
- Tangram tends to be slightly more confident on rare immune populations (T cells), sometimes at the cost of false positives in clearly stromal regions.
- For downstream niche analysis we default to cell2location because its uncertainty interval lets us threshold high-confidence calls.